In [ ]:
ayudame buscando errores en este algoritmo:
# Parameters
capital               = 1000
patrimony             = capital # Equity: tiene en cuenta la resta del entry_amount
risk_per_trade        = 0.06
fee_rate              = 0.0007
slippage_pct          = 0.0005
atr_period            = 14
atr_sl                = 2       # Value for SL
multiplier_tp         = 2       # Value for TP -> atr_tp = atr_sl x multiplier_tp
trailing_atr_mult     = 1.5       # Value por trailing stop
min_holding_period    = 0
leverage              = 100     # 10 = 1:10
use_trailing_stop     = True
use_breakeven         = use_trailing_stop
trailing_activation_r = 1.6     # solo activar trailing después de 2R de ganancia
breakeven_trigger_r   = 1     # break-even al llegar a 1R
# ── Gestión de riesgo de cartera ─────────────────────────────
# max_dd_daily_pct       = 0.05        # parar si el día pierde >5%
# cooldown_bars          = 3           # barras de espera tras un SL
# max_trades_per_day     = 6

profit_total = 0
loss_total = 0
position_opened = False
candle_trade = []
state = []
idx = 0
bars_held = np.nan
n = len(df)

for i in range(n): # (6, 45)

    bar        = df.iloc[i]
    prev_bar   = df.iloc[i - 1]
    signal     = bar["signal"]
    high       = bar["High"]
    low        = bar["Low"]
    atr        = bar["ATR"]
    price_var  = "Close"
    price      = bar[price_var]

    #-------------- 1. ENTRY
    if not position_opened and signal != 0:

        if signal == 1: 
            entry_price = price * (1 + slippage_pct)
        else:
            entry_price = price * (1 - slippage_pct)

        sl_distance = atr_sl * atr

        if signal == 1:
            stop_loss = entry_price - sl_distance
        else:
            stop_loss = entry_price + sl_distance

        tp_distance = abs(entry_price - stop_loss) * multiplier_tp

        if signal == 1:
            take_profit = entry_price + tp_distance
        else:
            take_profit = entry_price - tp_distance

        amount = patrimony * risk_per_trade
        patrimony = patrimony - amount

        risk_unit = abs(entry_price - stop_loss)
        size = amount / risk_unit
        position_value = size * entry_price
        max_position_value = amount * leverage      
        if position_value > max_position_value:
            size = max_position_value / entry_price

        # Niveles auxiliares (se calculan UNA vez en ENTRY)
        r_unit        = abs(entry_price - stop_loss)
        be_price      = entry_price + breakeven_trigger_r * r_unit * signal
        # partial_price = entry_price + partial_tp_r        * r_unit * signal

        bars_held = 0

        temp = {
            "id": [idx],
            "time": [bar.name],
            "signal": [signal],
            "signal_trade": [signal],
            "close": [entry_price],
            "entry_price": [entry_price],
            "sl": [stop_loss],
            "sl_inicial":     [stop_loss],   # guardamos el SL original para calcular R
            # "tstop": [stop_loss],
            "tp": [take_profit],
            "size": size,
            "bars_held": [bars_held],
            "entry_amount":[amount],
            "be_price":       [be_price],
            "breakeven_hit":  [False],
            "partial_hit":    [False],
            "trailing_active":[False],
            "price_prev":[prev_bar[price_var]],
            "diff":[bar[price_var] - prev_bar[price_var]],
            "delta":[(bar[price_var] - prev_bar[price_var])/prev_bar[price_var]],
            "diff_open": [np.nan],
            "delta_open":[np.nan],  
            "patrimony": [patrimony],
            "state": ["ENTRY"]
        }
        temp = pd.DataFrame(temp)
        state.append(temp)
        position_opened = True
        candle_trade = temp.copy()


    #-------------- 2. MANAGE TRADE
    if position_opened:
        bars_held += 1
        _exit        = False
        exitintrail  = False
        reason       = np.nan

        _close = bar["Close"]
        _sl = candle_trade["sl"][0]
        _tp = candle_trade["tp"][0]
        _entry_amount = candle_trade["entry_amount"][0]
        _entry_price = candle_trade["entry_price"][0]
        _size = candle_trade["size"][0]        
        signal_trade    = candle_trade["signal_trade"][0]
        _sl_inicial     = candle_trade["sl_inicial"][0]
        _be_price       = candle_trade["be_price"][0]
        # _partial_price  = candle_trade["partial_price"][0]
        _breakeven_hit  = candle_trade["breakeven_hit"][0]
        # _partial_hit    = candle_trade["partial_hit"][0]
        _trailing_active= candle_trade["trailing_active"][0]
        _close          = bar["Close"]



        temp = candle_trade.copy()

        temp["diff_open"] = (bar[price_var] - candle_trade[price_var.lower()])[0]
        temp["delta_open"] = ((bar[price_var] - candle_trade[price_var.lower()]) /
                              candle_trade[price_var.lower()])[0]
        
        # _actual_position_value = candle_trade["size"][0] * candle_trade["entry_price"][0]  

        # if temp["signal_trade"][0] == 1:
        #     pnl = _actual_position_value * temp["delta_open"][0]   
        # if temp["signal_trade"][0] == -1:
        #     pnl = _actual_position_value * (-temp["delta_open"][0])  

        # signal_trade = candle_trade["signal_trade"][0]

        # _exit = False
        # reason = np.nan

        # # Update trailing stop
        # ganancia_en_r = abs(_close - _entry_price) / abs(_entry_price - _sl)
            
        # if use_trailing_stop and ganancia_en_r >= trailing_activation_r:
        #     trail_dist = trailing_atr_mult * atr   # ej: 1.5 × $500 = $750
        #     exitintrail = False
        #     # BUY
        #     if (signal_trade == 1) and (new_trail > _sl):
        #         new_trail = _close - trail_dist        #  $50,000 - $750 = $49,250
        #         _sl = new_trail
        #         candle_trade["sl"] = _sl
        #         exitintrail = True
        #         pass
        #     # SELL
        #     if (signal_trade == -1) and (new_trail < _sl):
        #         new_trail = _close + trail_dist 
        #         _sl = new_trail
        #         candle_trade["sl"] = _sl
        #         exitintrail = True
        #         pass


        
        # P&L actual de la posición (sobre valor real apalancado)
        actual_pos_value = _size * _entry_price          
        delta_open       = (_close - _entry_price) / _entry_price
        pnl              = actual_pos_value * delta_open * signal_trade

        # ── BREAK-EVEN ───────────────────────────────────────
        if use_breakeven and not _breakeven_hit:
            be_hit = (signal_trade ==  1 and _close >= _be_price) or \
                     (signal_trade == -1 and _close <= _be_price)
            if be_hit:
                _sl = _entry_price
                candle_trade["sl"]           = _sl
                candle_trade["breakeven_hit"]= True
                _breakeven_hit               = True

        # ── TRAILING STOP (solo se activa después de trailing_activation_r) ──
        if use_trailing_stop:
            r_unit       = abs(_entry_price - _sl_inicial)
            ganancia_en_r = abs(_close - _entry_price) / r_unit if r_unit != 0 else 0

            if ganancia_en_r >= trailing_activation_r:
                candle_trade["trailing_active"] = True
                _trailing_active = True

            if _trailing_active:
                trail_dist = trailing_atr_mult * atr

                if signal_trade == 1:                     
                    new_trail = _close - trail_dist
                    if new_trail > _sl:
                        _sl = new_trail
                        candle_trade["sl"] = _sl
                        exitintrail = True
                else:
                    new_trail = _close + trail_dist
                    if new_trail < _sl:
                        _sl = new_trail
                        candle_trade["sl"] = _sl
                        exitintrail = True


        # Verify that the price crosses tp or tp
        if signal_trade == 1:
            if (_close >= _tp) or (_close <= _sl):
                _exit = True
                reason = "TP" if _close >= _tp else "SL"
                reason = "TS" if reason == "SL" and exitintrail else reason

        if signal_trade == -1:
            if (_close <= _tp) or (_close >= _sl):
                _exit = True
                reason = "TP" if _close <= _tp else "SL"
                

        # EXIT TRADE
        if _exit:
            exit_price = _tp if reason == "TP" else _sl

            if signal_trade == 1:
                _return = (exit_price - _entry_price) / _entry_price
            else:
                _return = (_entry_price - exit_price) / _entry_price
            
            actual_position_value = _size * _entry_price          
            gross_profit = _return * actual_position_value          
            fees         = actual_position_value * fee_rate * 2       
            profit_f     = gross_profit - fees
            patrimony    += _entry_amount + profit_f

            temp = {
                "id": [idx],
                "time": [bar.name],
                "close": [price],
                "signal_trade":[0],
                "price_prev":prev_bar[price_var],
                "diff":bar[price_var] - prev_bar[price_var],
                "delta":[(bar[price_var] - prev_bar[price_var])/prev_bar[price_var]],
                "patrimony": [patrimony],
                "state":"EXIT",
                "reason":[reason],
                "pnl":[_entry_amount + profit_f],
                "entry_amount":[_entry_amount],
                "profit_ex":[profit_f],
                "return_ex":[_return],
                "bars_held_ex":[bars_held]
            }

            temp = pd.DataFrame(temp)
            state.append(temp)

            position_opened = False
            candle_trade = []

        # 3. HOLD - NOT EXIT
        else:
            temp = candle_trade.copy()
            temp["id"] = idx
            temp["signal"] = np.nan
            temp["close"] = price
            temp["time"] = bar.name
            temp["bars_held"] = [bars_held]
            temp["price_prev"] = prev_bar[price_var]
            temp["diff"] = bar[price_var] - prev_bar[price_var]
            temp["delta"] = (bar[price_var] - prev_bar[price_var]) / prev_bar[price_var]
            temp["diff_open"] = (bar[price_var] - candle_trade[price_var.lower()])[0]
            temp["delta_open"] = ((bar[price_var] - candle_trade[price_var.lower()]) / candle_trade[price_var.lower()])[0]
            temp["patrimony"] = patrimony
            temp["state"] = "HOLD"
            temp["pnl"] = pnl

            state.append(temp)


    # 5. HOLD IF NOT SIGNAL AND TRADE
    if not position_opened and signal == 0:
        temp = {
            "id": [idx],
            "time": [bar.name],
            "signal_trade": [signal],
            "close": [price],
            "price_prev":prev_bar[price_var],
            "diff":bar[price_var] - prev_bar[price_var],
            "delta":[(bar[price_var] - prev_bar[price_var])/prev_bar[price_var]],
            "patrimony": [patrimony],
            "state":"NO SIGNAL"
        }
        temp = pd.DataFrame(temp)
        state.append(temp)

    idx += 1

state = pd.concat(state)
state.drop_duplicates("time", inplace = True) # Ajustar

# Devolviendo capital invertido a la ultima vela si finalizando se tiene una operacion abierta y esta no cerró
last = state.iloc[-1]
indeter_signal = False
if last["state"] == "HOLD":
    indeter_signal = True
    state["patrimony"] = np.where(state["id"] == last["id"], state["patrimony"] + state["entry_amount"], state["patrimony"])

if "profit_ex" not in state:
    state["profit_ex"] = np.nan
if "return_ex" not in state:
    state["return_ex"] = np.nan

state["profit"] = state["pnl"] - state["entry_amount"]
state["return%"] = state["profit"] / state["entry_amount"]
state["profit"] = np.where(state["state"] == "EXIT", state["profit_ex"], state["profit"])
state["return%"] = np.where(state["state"] == "EXIT", state["return_ex"], state["return%"])
state["peak"] = state["patrimony"].cummax()
state["drawdown"] = (state["patrimony"] - state["peak"]) / state["peak"]
exits = state[state["state"] == "EXIT"].copy()

returns = state.loc[state["state"] == "EXIT", "return%"].dropna()
returns_pct = exits["return%"]

if len(returns) > 1 and returns.std() != 0:
    sharpe = (returns.mean() / returns.std()) * np.sqrt(len(returns))
else:
    sharpe = np.nan

max_dd = state["drawdown"].min()

capital_f = state["patrimony"].iloc[-1]
pnl_f = capital_f - capital
return_f = ((capital_f / capital) - 1)*100
exits["return%"] = exits["return%"]*100
n_trades = exits.shape[0] if not indeter_signal else exits.shape[0] - 1
win_rate = (exits[(exits["profit"] > 0)].shape[0] / n_trades)*100
days = (state["time"].max() - state["time"].min()) / pd.Timedelta(days=1)
downside = exits["return%"]
downside = downside[downside < 0]
sortino = (exits["return%"].mean() / downside.std()) * np.sqrt(24*365) 
gains = exits[exits["profit"] > 0]
loss = exits[exits["profit"] < 0]
exits["time_t1"] = exits["time"].shift(1)
exits["diff_time"] = (exits["time"] - exits["time_t1"]) / pd.Timedelta(days=1)
cagr   = ((capital_f / capital) ** (365 / max(days, 1)) - 1) * 100
calmar = cagr / abs(max_dd * 100) if max_dd != 0 else np.nan
win_rate_dec = len(gains) / n_trades if n_trades > 0 else 0
max_loss_streak, cur_streak = 0, 0
for s in exits["profit"]:
    cur_streak      = cur_streak + 1 if s < 0 else 0
    max_loss_streak = max(max_loss_streak, cur_streak)
max_gain_streak, cur_streak = 0, 0
for s in exits["profit"]:
    cur_streak      = cur_streak + 1 if s > 0 else 0
    max_gain_streak = max(max_gain_streak, cur_streak)
hold_bars  = state[state["state"] == "HOLD"].shape[0]
exposure   = hold_bars / len(state) * 100
var_95  = np.percentile(returns_pct, 5)  if len(returns_pct) > 0 else np.nan
cvar_95 = returns_pct[returns_pct <= var_95].mean() if len(returns_pct) > 0 else np.nan
recovery_factor = pnl_f / abs(max_dd * capital) if max_dd != 0 else np.nan
n_sl = exits[exits["reason"] == "SL"].shape[0]
n_tp = exits[exits["reason"] == "TP"].shape[0]

print("─────────────── Symbol ──────────────")
print(f"               {symb}")

print("\n───────── Time and Candles ──────────")
print("Candles:", f"{len(df)} | Days: {int(days)}") # {window}
print("Trades:           ", n_trades, f"| Wins: {len(gains)} | Loss: {len(loss)}")
print("Trades/day:       ", f"{(n_trades/days):.1f}")
print("Time/trade:       ", f"Mean: {exits["bars_held_ex"].mean():.0f} | Min: {exits["bars_held_ex"].min():.0f} | Max: {exits["bars_held_ex"].max():.0f}")
print("Time inter trades:", f"Mean: {exits["diff_time"].mean():.0f} | Min: {exits["diff_time"].min():.0f} | Max: {exits["diff_time"].max():.0f}")

print("\n─────────── Final Results ───────────")
print("Win rate:         ", f"{win_rate:.0f}%")
print("Capital initial:  ", f"${capital}")
print("Capital final:    ", f"${capital_f:.1f}")
print("PnL:              ", f"${pnl_f:.1f}")
print("Return:           ", f"{(return_f):.1f}%")
print("Max drawdown:     ", f"{max_dd:.1%}\n")

print("Sharpe:           ", f"{sharpe:.1f}")
print("Sortino:          ", f"{sortino:.0f}")
print("Calmar:           ", f"{calmar:.0f}")
print("CAGR:             ", f"{cagr:.0f}\n")

print("Total gains:      ", f"${gains["profit"].sum():.0f}")
print("Total loss:       ", f"${loss["profit"].sum():.0f}")
print(" ")
print("Return/trade:     ", f"Mean: {exits["return%"].mean():.0f}% | Min: {exits["return%"].min():.0f}% | Max: {exits["return%"].max():.0f}%")
print("Profit/trade:     ", f"Mean: ${exits["profit"].mean():.1f} | Min: ${exits["profit"].min():.1f} | Max: ${exits["profit"].max():.1f}")

print("\n─────────── Metrics Risk ───────────") # Risk:Reward Ratio
exits["return_on_margin"] = exits["profit"] / exits["entry_amount"] * 100 # Retorno promedio real sobre margen (no sobre precio)
avg_win = exits[exits['profit']>0]['return%'].mean()
avg_loss = exits[exits['profit']<0]['return%'].mean()
avg_win_margin = exits[exits['profit']>0]['return_on_margin'].mean()
avg_loss_margin = exits[exits['profit']<0]['return_on_margin'].mean()
loss_rate = 1 - win_rate   
prom_fee = exits['entry_amount'].mean() * leverage * fee_rate * 2
expectancy = (win_rate * avg_win) - (loss_rate * avg_loss) # Para ser rentable que sea > 0. Te dice cuánto esperas ganar o perder por operación, en 
# expectancy = win_rate_dec * avg_win + (1 - win_rate_dec) * avg_loss
# promedio. Expectancy = ganancia promedio por trade

print(f"Expectancy:       {expectancy:.1f}")
print(f"Avg win (price):  {avg_win:.1f}%")
print(f"Avg loss (price): {avg_loss:.1f}%\n")

print(f"Ratio W/L:        {abs(avg_win / avg_loss):.1f}x | Gain prom = {abs(avg_win / avg_loss):.1f} x Loss") # en promedio cada ganancia es 2.6 veces más grande que cada pérdida.
print(f"Recovery factor:  {recovery_factor:.2f}")
print(f"VaR 95%:          {var_95:.1f}%")
print(f"CVaR 95%:         {cvar_95:.1f}%\n")

print(f"Max racha loss:   {max_loss_streak:.0f}")
print(f"Max racha gain:   {max_gain_streak:.0f}\n")

print(f"Closed x TP:      {n_tp} ({n_tp/n_trades*100:.0f}%)")
print(f"Closed x SL:      {n_sl} ({n_sl/n_trades*100:.0f}%)\n")

print(f"Avg win (margin):   {avg_win_margin:.0f}%")
print(f"Avg loss (margin):  {avg_loss_margin:.0f}%\n")

print(f"Fees estim/trade:   ${prom_fee:.1f}")
print(f"Total fees estim:   ${prom_fee * n_trades:.0f}")
print(f"Fees as %avg_win:   {prom_fee / exits[exits['profit']>0]['profit'].mean() * 100:.0f}%")

state.drop(columns = ["diff_open", "price_prev", "diff", "delta", "profit_ex", "return_ex", "peak", "bars_held_ex", "delta_open", "be_price", "breakeven_hit",
                      "trailing_active", "sl_inicial"], inplace =True)
state2 = state.copy()